# MEPI v1 — environment check

**Purpose:** fail-closed readiness check before any training notebook.  
**Inputs:** project source, frozen protocol, Python/PyTorch/CUDA environment, project tests.  
**Outputs:** visible PASS/FAIL output and `reports/notebook_workflow_test_state.json`.  
**Frozen protocol:** MEPI-FROZEN-PROTOCOL v1.1.  
**Trains a model:** no.  
**Checkpoints/results:** no model checkpoint; test state is saved under `reports/`.

In [ ]:
from pathlib import Path

# MEPI Run-All configuration. Safe recovery is the default.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "mepi_v1").is_dir():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
SEED = 42
FORCE_RETRAIN = False
AUTO_RESUME = True
CHECKPOINT_EVERY_N_STEPS = 1000

print(f"[PASS] Configuration loaded: PROJECT_ROOT={PROJECT_ROOT}, SEED={SEED}")
print(f"FORCE_RETRAIN={FORCE_RETRAIN}; AUTO_RESUME={AUTO_RESUME}; checkpoint interval={CHECKPOINT_EVERY_N_STEPS} steps")


## Load readiness checks

Import the shared checker from the repository source.

In [ ]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.mepi_v1.notebook_workflow import run_environment_checks
print("[PASS] Project source imports completed")


## Run all assertions and tests

Every assertion is printed. A failed test or missing CUDA raises an informative exception and stops Run All.

In [ ]:
environment_summary = run_environment_checks(PROJECT_ROOT, require_cuda=True, run_tests=True)


## Final readiness summary

Display the final state only when protocol assertions, project tests, and CUDA have passed.

In [ ]:
import pandas as pd
display(pd.DataFrame([{'environment_ready': environment_summary['environment_ready'], 'cuda_ready': environment_summary['cuda_ready'], 'tests': environment_summary['tests']['status'], 'protocol_assertions': environment_summary['protocol_assertions']}]))
print("ENVIRONMENT CHECK: PASS")
